# SonicArm V9 第一阶段软件预集成

本 Notebook 使用既有 FPGA-8 Overlay，将同一段16 kHz录音送入 CAM++ 声纹和 PS TC-ResNet2，并更新网页 UI。

**安全边界：不导入串口库、不打开串口、`execution_enabled=false`，机械臂应保持断开。**

本阶段的电脑浏览器仅用于调试，不是最终显示验收。最终必须在FPGA-9同一Overlay下完成PYNQ HDMI实体屏与USB HID触摸验证。


In [ ]:
from pathlib import Path
import json, subprocess, sys, time

HERE = Path.cwd().resolve()
assert (HERE / 'run_validation.py').is_file(), '请在 integration_v9 目录打开本 Notebook'
NAME = '张同学'
USER_ID = 'SA-002'
print('工作目录:', HERE)
print('请按实际人员修改 NAME 和 USER_ID；机械臂保持断开。')


## 启动本地 UI

若 UI 已经在8080端口运行，请不要重复执行本单元。


In [ ]:
ui_log = open(HERE / 'ui_server.log', 'a', encoding='utf-8')
ui_process = subprocess.Popen(
    [sys.executable, str(HERE / 'ui' / 'server.py'), '--host', '0.0.0.0', '--port', '8080'],
    cwd=str(HERE / 'ui'), stdout=ui_log, stderr=subprocess.STDOUT
)
time.sleep(1.0)
print('UI PID:', ui_process.pid)
print('中间调试可访问 http://<PYNQ-IP>:8080/auth.html；这不构成最终实体屏验收')


## 执行一轮集成验证

依次采集2秒安静基线和5秒完整命令。身份通过后才运行短语模型，但任何结果都不会发送到机械臂。


In [ ]:
completed = subprocess.run(
    [sys.executable, str(HERE / 'run_validation.py'), '--name', NAME, '--user-id', USER_ID],
    cwd=str(HERE), text=True
)
print('返回码:', completed.returncode)
if completed.returncode != 0:
    print('请查看 results/run_<时间>/result.json 和 ui_server.log')


## 查看最新结果


In [ ]:
runs = sorted((HERE / 'results').glob('run_*'))
assert runs, '尚无运行结果'
latest = runs[-1] / 'result.json'
print(latest)
print(latest.read_text(encoding='utf-8'))


## 完成后停止 UI 服务


In [ ]:
if 'ui_process' in globals() and ui_process.poll() is None:
    ui_process.terminate()
    ui_process.wait(timeout=5)
if 'ui_log' in globals() and not ui_log.closed:
    ui_log.close()
print('UI服务已停止；Overlay不会在此单元重新加载。')
